# Experiment 8: Clustering Human Activity Recognition Data
## K-Means, DBSCAN, and Hierarchical Clustering

**Objective:** Implement and analyze K-Means, DBSCAN, and Hierarchical Agglomerative Clustering (HAC) on the Human Activity Recognition (HAR) Using Smartphones dataset, visualize clusters, and compare against ground-truth activity labels.

**Dataset:** UCI "Human Activity Recognition Using Smartphones" dataset — 30 volunteers, 6 activities (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING), 561 time/frequency-domain features per 2.56s window (128 readings, 50% overlap).


In [10]:
# 1. Imports
import warnings
warnings.filterwarnings('ignore')

import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              calinski_harabasz_score, adjusted_rand_score,
                              normalized_mutual_info_score, confusion_matrix)
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Download and Load the Dataset

The dataset is downloaded directly from the UCI Machine Learning Repository. This requires internet access; if the download fails, manually download the "UCI HAR Dataset.zip" from
https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones
and place it in the working directory before re-running this cell.


In [ ]:
DATA_DIR = 'UCI_HAR_Dataset'
ZIP_PATH = 'UCI_HAR_Dataset.zip'
URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'

if not os.path.exists(DATA_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Downloading dataset...")
        try:
            urllib.request.urlretrieve(URL, ZIP_PATH)
            print("Download complete.")
        except Exception as e:
            print(f"Download failed: {e}")
            print("Please manually download the dataset zip and place it as 'UCI_HAR_Dataset.zip' in this directory.")
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall('.')
        # The archive typically extracts to a folder named 'UCI HAR Dataset'
        if os.path.exists('UCI HAR Dataset') and not os.path.exists(DATA_DIR):
            os.rename('UCI HAR Dataset', DATA_DIR)
        print("Extraction complete.")

print("Contents:", os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "DATASET NOT FOUND")


NotADirectoryError: [WinError 267] The directory name is invalid: 'train.csv'

In [ ]:
# Load feature names, train and test splits, merge into one dataset for clustering
base = DATA_DIR

features = pd.read_csv(os.path.join(base, 'features.txt'), sep=r'\s+', header=None, names=['idx','feature'])
feature_names = features['feature'].tolist()

# Deduplicate feature names (the raw file has some duplicates)
seen = {}
unique_names = []
for name in feature_names:
    if name in seen:
        seen[name] += 1
        unique_names.append(f"{name}_{seen[name]}")
    else:
        seen[name] = 0
        unique_names.append(name)
feature_names = unique_names

X_train = pd.read_csv(os.path.join(base, 'train', 'X_train.txt'), sep=r'\s+', header=None, names=feature_names)
y_train = pd.read_csv(os.path.join(base, 'train', 'y_train.txt'), sep=r'\s+', header=None, names=['activity_id'])
subj_train = pd.read_csv(os.path.join(base, 'train', 'subject_train.txt'), sep=r'\s+', header=None, names=['subject'])

X_test = pd.read_csv(os.path.join(base, 'test', 'X_test.txt'), sep=r'\s+', header=None, names=feature_names)
y_test = pd.read_csv(os.path.join(base, 'test', 'y_test.txt'), sep=r'\s+', header=None, names=['activity_id'])
subj_test = pd.read_csv(os.path.join(base, 'test', 'subject_test.txt'), sep=r'\s+', header=None, names=['subject'])

activity_labels = pd.read_csv(os.path.join(base, 'activity_labels.txt'), sep=r'\s+', header=None, names=['id','activity'])
id_to_activity = dict(zip(activity_labels['id'], activity_labels['activity']))

X = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
y = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)
subjects = pd.concat([subj_train, subj_test], axis=0).reset_index(drop=True)
y['activity'] = y['activity_id'].map(id_to_activity)

print("Full dataset shape:", X.shape)
print("Number of subjects:", subjects['subject'].nunique())
print("\nActivity distribution:")
print(y['activity'].value_counts())


FileNotFoundError: [Errno 2] No such file or directory: 'train\\features.txt'

## 3. Preprocessing

In [ ]:
# Check for missing values
print("Total missing values:", X.isnull().sum().sum())

# Features are already normalized to [-1,1] in the raw dataset, but we standardize
# again to ensure zero mean / unit variance for distance-based clustering algorithms.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled data shape:", X_scaled.shape)

# To keep runtime manageable for clustering + t-SNE on this ~10,299-sample, 561-feature
# dataset, we optionally work with a stratified subsample. Set SUBSAMPLE=None to use all data.
SUBSAMPLE = 3000

if SUBSAMPLE is not None and SUBSAMPLE < X_scaled.shape[0]:
    rng = np.random.RandomState(RANDOM_STATE)
    idx = y.groupby('activity', group_keys=False).apply(
        lambda g: pd.Series(g.index)
    ).values
    _, sample_idx = train_test_split_idx = (None, None)
    from sklearn.model_selection import train_test_split
    sample_idx, _ = train_test_split(
        np.arange(X_scaled.shape[0]),
        train_size=SUBSAMPLE,
        stratify=y['activity'],
        random_state=RANDOM_STATE
    )
    X_use = X_scaled[sample_idx]
    y_use = y.iloc[sample_idx].reset_index(drop=True)
else:
    X_use = X_scaled
    y_use = y.reset_index(drop=True)

print("Data used for clustering:", X_use.shape)


## 4. Exploratory Data Analysis

Visualize feature distributions and apply PCA/t-SNE for 2D visualization of the raw (labeled) structure before clustering.


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(y='activity', data=y_use, order=y_use['activity'].value_counts().index)
plt.title('Activity Class Distribution (sample used)')
plt.xlabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of a few example features
sample_features = feature_names[:4]
fig, axes = plt.subplots(1, 4, figsize=(18,4))
for ax, feat in zip(axes, sample_features):
    ax.hist(X[feat], bins=40, color='steelblue')
    ax.set_title(feat, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# PCA to 2D for visualization
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca2 = pca_2d.fit_transform(X_use)

plt.figure(figsize=(7,6))
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=y_use['activity'], palette='tab10', s=12, alpha=0.7)
plt.title('PCA (2D) — Colored by True Activity Label')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print(f"Explained variance (2 components): {pca_2d.explained_variance_ratio_.sum()*100:.2f}%")


In [ ]:
# t-SNE to 2D for visualization (this can take a minute)
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, init='pca', learning_rate='auto')
X_tsne2 = tsne.fit_transform(X_use)

plt.figure(figsize=(7,6))
sns.scatterplot(x=X_tsne2[:,0], y=X_tsne2[:,1], hue=y_use['activity'], palette='tab10', s=12, alpha=0.7)
plt.title('t-SNE (2D) — Colored by True Activity Label')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


## 5. Model A — K-Means Clustering

### 5.1 Elbow Method + Silhouette Score to choose k


In [ ]:
k_values = list(range(2, 9))
wcss = []
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_use)
    wcss.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_use, labels))
    print(f"k={k}: WCSS={km.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.4f}")

elbow_table = pd.DataFrame({
    'Number of Clusters (k)': k_values,
    'WCSS (Inertia)': [round(w,2) for w in wcss],
    'Silhouette Score': [round(s,4) for s in silhouette_scores]
})
elbow_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].plot(k_values, wcss, marker='o')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_title('Elbow Method: k vs WCSS')
axes[0].grid(alpha=0.3)

axes[1].plot(k_values, silhouette_scores, marker='o', color='darkorange')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('k vs Silhouette Score')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_k_silhouette = k_values[int(np.argmax(silhouette_scores))]
print(f"k with highest silhouette score: {best_k_silhouette}")
print("Note: the dataset has 6 true activity classes; compare the elbow/silhouette-suggested k "
      "against k=6 in the analysis below.")


In [ ]:
# Justification / selection of k
# Students: inspect the elbow curve above and the silhouette scores, then set K_CHOSEN.
K_CHOSEN = 6  # chosen to match the number of true activity classes; adjust based on elbow/silhouette results if needed

kmeans_final = KMeans(n_clusters=K_CHOSEN, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_use)

print(f"Chosen k = {K_CHOSEN}")
print(f"Justification: k=6 matches the number of ground-truth activities; the elbow curve shows "
      f"diminishing WCSS reduction beyond k~{best_k_silhouette}-6, and silhouette score peaks around k={best_k_silhouette}.")


### 5.2 Visualize K-Means Clusters (PCA / t-SNE projection)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=kmeans_labels, palette='tab10', s=12, alpha=0.7, ax=axes[0], legend='full')
axes[0].set_title('K-Means Clusters (PCA projection)')

sns.scatterplot(x=X_tsne2[:,0], y=X_tsne2[:,1], hue=kmeans_labels, palette='tab10', s=12, alpha=0.7, ax=axes[1], legend='full')
axes[1].set_title('K-Means Clusters (t-SNE projection)')

plt.tight_layout()
plt.show()


## 6. Model B — DBSCAN

### 6.1 Tuning eps using the k-distance (nearest-neighbor) plot


In [ ]:
min_pts = 2 * X_use.shape[1]  # common heuristic: minPts >= 2*dimensions; we'll also try smaller values
min_pts_try = 10  # more practical value for visualization / actual clustering runs

neighbors = NearestNeighbors(n_neighbors=min_pts_try)
neighbors_fit = neighbors.fit(X_use)
distances, indices = neighbors_fit.kneighbors(X_use)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(7,5))
plt.plot(k_distances)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{min_pts_try}-NN distance')
plt.title('K-Distance Plot for DBSCAN eps Selection')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Look for the 'elbow'/knee in this curve — that distance value is a good starting point for eps.")


In [ ]:
# Students: set EPS based on the knee observed in the k-distance plot above.
EPS = 8.0        # adjust based on the k-distance plot
MIN_SAMPLES = min_pts_try

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(X_use)

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = np.sum(dbscan_labels == -1)

print(f"DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})")
print(f"Number of clusters found: {n_clusters_db}")
print(f"Number of noise points: {n_noise} ({n_noise/len(dbscan_labels)*100:.2f}% of data)")
print("\nCluster size distribution:")
print(pd.Series(dbscan_labels).value_counts())


In [ ]:
# Try a small grid of eps/min_samples to show sensitivity (Table for report)
eps_grid = [5, 6, 7, 8, 9, 10]
minpts_grid = [5, 10, 15]

dbscan_results = []
for eps in eps_grid:
    for mp in minpts_grid:
        labels = DBSCAN(eps=eps, min_samples=mp).fit_predict(X_use)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct = np.mean(labels == -1) * 100
        sil = silhouette_score(X_use, labels) if n_clust > 1 and n_clust < len(labels) else np.nan
        dbscan_results.append({'eps': eps, 'min_samples': mp, 'n_clusters': n_clust,
                                'noise_%': round(noise_pct,2), 'silhouette': round(sil,4) if not np.isnan(sil) else None})

dbscan_grid_table = pd.DataFrame(dbscan_results)
dbscan_grid_table


### 6.2 Visualize DBSCAN Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=dbscan_labels, palette='tab10', s=12, alpha=0.7, ax=axes[0], legend='full')
axes[0].set_title(f'DBSCAN Clusters (PCA projection)\neps={EPS}, min_samples={MIN_SAMPLES} — label -1 = noise')

sns.scatterplot(x=X_tsne2[:,0], y=X_tsne2[:,1], hue=dbscan_labels, palette='tab10', s=12, alpha=0.7, ax=axes[1], legend='full')
axes[1].set_title('DBSCAN Clusters (t-SNE projection)')

plt.tight_layout()
plt.show()


## 7. Model C — Hierarchical Agglomerative Clustering (HAC)

### 7.1 Dendrogram (Ward's linkage)


In [ ]:
# For dendrogram readability, use a smaller subsample
DENDRO_SAMPLE = 200
rng = np.random.RandomState(RANDOM_STATE)
dendro_idx = rng.choice(X_use.shape[0], size=min(DENDRO_SAMPLE, X_use.shape[0]), replace=False)
X_dendro = X_use[dendro_idx]

Z = linkage(X_dendro, method='ward')

plt.figure(figsize=(14,6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=9., show_contracted=True)
plt.title("Hierarchical Clustering Dendrogram (Ward's Linkage, sample of 200 points)")
plt.xlabel('Cluster size / sample index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()


### 7.2 Compare Linkage Criteria

In [ ]:
linkage_methods = ['single', 'complete', 'average', 'ward']
linkage_results = []

for method in linkage_methods:
    hac = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage=method)
    labels = hac.fit_predict(X_use)
    sil = silhouette_score(X_use, labels)
    db = davies_bouldin_score(X_use, labels)
    ch = calinski_harabasz_score(X_use, labels)
    ari = adjusted_rand_score(y_use['activity_id'], labels)
    nmi = normalized_mutual_info_score(y_use['activity_id'], labels)
    linkage_results.append({
        'Linkage': method, 'Silhouette': round(sil,4), 'Davies-Bouldin': round(db,4),
        'Calinski-Harabasz': round(ch,2), 'ARI': round(ari,4), 'NMI': round(nmi,4)
    })

linkage_table = pd.DataFrame(linkage_results)
print(f"Hierarchical Clustering — Linkage Comparison (n_clusters={K_CHOSEN})")
linkage_table


In [ ]:
# Final HAC model using Ward's linkage (best-practice default for compact clusters)
hac_final = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage='ward')
hac_labels = hac_final.fit_predict(X_use)


### 7.3 Visualize Hierarchical Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,6))

sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=hac_labels, palette='tab10', s=12, alpha=0.7, ax=axes[0], legend='full')
axes[0].set_title(f"Hierarchical (Ward) Clusters (PCA projection), k={K_CHOSEN}")

sns.scatterplot(x=X_tsne2[:,0], y=X_tsne2[:,1], hue=hac_labels, palette='tab10', s=12, alpha=0.7, ax=axes[1], legend='full')
axes[1].set_title('Hierarchical (Ward) Clusters (t-SNE projection)')

plt.tight_layout()
plt.show()


## 8. Evaluation Metrics — All Algorithms

Internal metrics (no ground truth needed): Silhouette Score, Davies–Bouldin Index, Calinski–Harabasz Index.
External metrics (using true activity labels): Adjusted Rand Index (ARI), Normalized Mutual Information (NMI).


In [ ]:
def evaluate_clustering(name, labels, X_data, y_true):
    # Filter out noise points (-1) for internal metrics that require >=2 clusters without noise-only degeneracy
    mask = labels != -2  # no-op mask; DBSCAN noise (-1) is kept as its own 'cluster' for these metrics
    n_unique = len(set(labels[mask]))

    if n_unique > 1 and n_unique < len(labels[mask]):
        sil = silhouette_score(X_data[mask], labels[mask])
        db = davies_bouldin_score(X_data[mask], labels[mask])
        ch = calinski_harabasz_score(X_data[mask], labels[mask])
    else:
        sil = db = ch = np.nan

    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)

    return {
        'Algorithm': name,
        'Silhouette Score': round(sil,4) if not np.isnan(sil) else None,
        'Davies-Bouldin Index': round(db,4) if not np.isnan(db) else None,
        'Calinski-Harabasz Index': round(ch,2) if not np.isnan(ch) else None,
        'Adjusted Rand Index (ARI)': round(ari,4),
        'Normalized Mutual Info (NMI)': round(nmi,4)
    }

results_summary = pd.DataFrame([
    evaluate_clustering('K-Means', kmeans_labels, X_use, y_use['activity_id'].values),
    evaluate_clustering('DBSCAN', dbscan_labels, X_use, y_use['activity_id'].values),
    evaluate_clustering('Hierarchical (Ward)', hac_labels, X_use, y_use['activity_id'].values),
])
results_summary


In [ ]:
# Bar plots comparing algorithms across metrics
metrics_to_plot = ['Silhouette Score', 'Davies-Bouldin Index', 'Adjusted Rand Index (ARI)', 'Normalized Mutual Info (NMI)']

fig, axes = plt.subplots(2, 2, figsize=(12,9))
axes = axes.flatten()

for ax, metric in zip(axes, metrics_to_plot):
    ax.bar(results_summary['Algorithm'], results_summary[metric].astype(float), color=['#4C72B0','#DD8452','#55A868'])
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## 9. Confusion Matrices (Clusters vs. True Activities)

In [ ]:
def plot_cluster_activity_crosstab(labels, title, ax):
    ct = pd.crosstab(pd.Series(labels, name='Cluster'), y_use['activity'])
    sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(20,6))
plot_cluster_activity_crosstab(kmeans_labels, 'K-Means: Cluster vs Activity', axes[0])
plot_cluster_activity_crosstab(dbscan_labels, 'DBSCAN: Cluster vs Activity (-1 = noise)', axes[1])
plot_cluster_activity_crosstab(hac_labels, 'Hierarchical (Ward): Cluster vs Activity', axes[2])
plt.tight_layout()
plt.show()


## 10. Observation Questions

Use the metric tables and plots above as evidence when answering.


In [ ]:
print("Quantitative summary for observations:\n")
print(results_summary.to_string(index=False))
print("\nDBSCAN noise points:", n_noise, f"({n_noise/len(dbscan_labels)*100:.2f}% of data)")
print("\nLinkage comparison (Hierarchical):")
print(linkage_table.to_string(index=False))


## 11. Final Conclusion

_Summarize, using the evidence generated above:_
- Which of K-Means, DBSCAN, and Hierarchical Clustering best recovered the true activity structure (highest ARI/NMI) and why.
- The practical difficulty DBSCAN faces on high-dimensional HAR features (curse of dimensionality affecting density estimates) versus its strength on datasets with arbitrary-shaped clusters and noise.
- The trade-off between K-Means' simplicity/speed and Hierarchical Clustering's richer dendrogram-based interpretability.
- Practical recommendation: for HAR-like tabular sensor-feature data, K-Means or Ward-linkage Hierarchical Clustering (with PCA for visualization) tend to be more reliable starting points than DBSCAN unless density parameters are carefully tuned per-feature-space.
